###  **Evaluación integral de RAG avanzado y agentes con verificación**

#### **Propósito técnico**

Este cuaderno cierra los temas relacionados  integrando LLMs, RAG, búsqueda híbrida, reranking, verificación, guardrails, evaluación y análisis de fallos.

El objetivo no es construir un sistema grande, sino un laboratorio pequeño y reproducible donde se observe la diferencia entre un RAG básico y un RAG avanzado. El cuaderno usa un corpus controlado, preguntas con respuestas esperadas y métricas simples para diagnosticar si el error aparece en la recuperación, en el contexto enviado al modelo o en la generación final.

La idea principal es que un sistema con LLM no debe evaluarse solo por una respuesta que parece correcta. Se debe verificar si recuperó evidencia adecuada, si la respuesta está fundamentada, si las citas son válidas y si los guardrails evitan salidas inseguras o no sustentadas.

#### **1. Configuración inicial**


En esta sección se importan las bibliotecas necesarias y se definen utilidades generales. Para mantener el cuaderno ejecutable en CPU y sin servicios externos, se usan recuperadores simples basados en tokens y vectores TF-IDF. En un sistema real, estas piezas podrían reemplazarse por `sentence-transformers`, FAISS, Qdrant, Milvus, OpenSearch, un cross-encoder de reranking y un LLM local o remoto.

La finalidad de esta celda no es competir con modelos industriales, sino mostrar la arquitectura de evaluación: documentos, recuperación, fusión, reranking, generación, verificación y diagnóstico.

In [ ]:
from __future__ import annotations

from dataclasses import dataclass, field
from typing import Any, Dict, List, Optional
from collections import Counter, defaultdict
import math
import re
import time

#### **2. Corpus controlado y benchmark**

Un sistema RAG se evalúa mejor cuando existe un conjunto pequeño de preguntas con evidencia esperada. Por eso se define un corpus con documentos breves sobre RAG, embeddings, BM25, reranking, guardrails, agentes y RLHF. Cada documento tiene un `id`, un título y un texto.

También se define un benchmark. Cada caso incluye la pregunta, los documentos correctos, una respuesta de referencia y términos obligatorios. Esto permite medir dos niveles: si el sistema recuperó los documentos correctos y si la respuesta final contiene información esencial.

In [ ]:
@dataclass
class Document:
    id: str
    title: str
    text: str
    metadata: Dict[str, Any] = field(default_factory=dict)


documents = [
    Document(
        id="doc_rag_01",
        title="RAG y evidencia externa",
        text=(
            "RAG conecta un LLM con documentos externos. "
            "Primero recupera evidencia relevante y luego genera una respuesta fundamentada. "
            "Esto reduce respuestas sin soporte documental."
        ),
        metadata={"tema": "rag", "nivel": "base"},
    ),
    Document(
        id="doc_embeddings_01",
        title="Embeddings y búsqueda semántica",
        text=(
            "Un embedding representa el significado de un texto como vector denso. "
            "La búsqueda semántica recupera documentos por similitud conceptual, "
            "aunque no compartan exactamente las mismas palabras."
        ),
        metadata={"tema": "embeddings", "nivel": "base"},
    ),
    Document(
        id="doc_hybrid_01",
        title="Búsqueda híbrida",
        text=(
            "BM25 recupera coincidencias exactas de términos y entidades. "
            "Los embeddings recuperan similitud semántica. "
            "En sistemas reales, BM25 y embeddings son complementarios."
        ),
        metadata={"tema": "hybrid_search", "nivel": "intermedio"},
    ),
    Document(
        id="doc_rerank_01",
        title="Reranking y contexto",
        text=(
            "El reranking reordena los documentos recuperados usando un criterio más preciso. "
            "Ayuda a colocar la evidencia más útil en las primeras posiciones. "
            "La compresión contextual reduce ruido antes de invocar al LLM."
        ),
        metadata={"tema": "reranking", "nivel": "intermedio"},
    ),
    Document(
        id="doc_guardrails_01",
        title="Guardrails y verificación",
        text=(
            "Los guardrails controlan entradas, herramientas, permisos, formato y salidas. "
            "La verificación comprueba si una respuesta está apoyada por evidencia, "
            "si respeta políticas y si evita información no fundamentada."
        ),
        metadata={"tema": "guardrails", "nivel": "intermedio"},
    ),
    Document(
        id="doc_agents_01",
        title="Agentes con herramientas",
        text=(
            "Un agente combina LLM, herramientas, memoria, planificación y verificación. "
            "El agente decide cuándo usar una herramienta, observa el resultado "
            "y responde cuando la evidencia es suficiente."
        ),
        metadata={"tema": "agentes", "nivel": "intermedio"},
    ),
    Document(
        id="doc_react_01",
        title="ReAct",
        text=(
            "ReAct organiza el trabajo del agente en ciclos de pensamiento operativo, "
            "acción y observación. Un ciclo puede buscar evidencia y otro puede calcular "
            "o verificar el resultado."
        ),
        metadata={"tema": "react", "nivel": "intermedio"},
    ),
    Document(
        id="doc_rlhf_01",
        title="RLHF y preferencias",
        text=(
            "RLHF usa preferencias humanas para alinear modelos. "
            "Un reward model aprende a puntuar respuestas preferidas sobre respuestas rechazadas. "
            "Luego la política puede ajustarse con métodos como PPO."
        ),
        metadata={"tema": "rlhf", "nivel": "avanzado"},
    ),
]


benchmark = [
    {
        "question": "¿Qué problema resuelve RAG?",
        "gold_docs": {"doc_rag_01"},
        "reference_answer": "RAG conecta el LLM con documentos externos para responder con evidencia y reducir respuestas sin fundamento.",
        "required_terms": ["documentos externos", "evidencia", "LLM"],
    },
    {
        "question": "¿Por qué BM25 y embeddings son complementarios?",
        "gold_docs": {"doc_hybrid_01", "doc_embeddings_01"},
        "reference_answer": "BM25 recupera coincidencias exactas y los embeddings recuperan similitud semántica.",
        "required_terms": ["BM25", "embeddings", "similitud semántica"],
    },
    {
        "question": "¿Para qué sirve el reranking en RAG avanzado?",
        "gold_docs": {"doc_rerank_01"},
        "reference_answer": "El reranking reordena documentos recuperados para colocar la evidencia más útil primero.",
        "required_terms": ["reranking", "evidencia", "primeras posiciones"],
    },
    {
        "question": "¿Qué controlan los guardrails?",
        "gold_docs": {"doc_guardrails_01"},
        "reference_answer": "Los guardrails controlan entradas, herramientas, permisos, formato y salidas.",
        "required_terms": ["entradas", "herramientas", "permisos", "salidas"],
    },
    {
        "question": "¿Cómo se relacionan agentes y herramientas?",
        "gold_docs": {"doc_agents_01", "doc_react_01"},
        "reference_answer": "Un agente decide cuándo usar herramientas, observa resultados y responde cuando la evidencia es suficiente.",
        "required_terms": ["agente", "herramientas", "observa", "evidencia"],
    },
]

#### **3. Normalización y representación de texto**


Toda recuperación necesita una forma de comparar preguntas y documentos. En sistemas reales, se usarían embeddings densos entrenados con contrastive learning, junto con índices vectoriales. Aquí se implementa una representación TF-IDF simple para que el cuaderno sea autocontenido.

La normalización convierte texto a minúsculas, elimina signos y separa tokens. Esta decisión afecta mucho a BM25, TF-IDF y cualquier recuperador basado en palabras. En producción, se debe considerar idioma, lematización, stopwords, entidades, símbolos técnicos y metadatos.

In [ ]:
def normalizeText(text: str) -> str:
    # Normaliza texto para comparación léxica.
    text = text.lower()
    text = re.sub(r"[^a-záéíóúñü0-9\s]", " ", text)
    return " ".join(text.split())


def tokenizeText(text: str) -> List[str]:
    # Convierte un texto normalizado en tokens.
    return normalizeText(text).split()


def termFrequency(tokens: List[str]) -> Counter:
    # Calcula frecuencia de términos para una lista de tokens.
    return Counter(tokens)


def buildVocabulary(docs: List[Document]) -> List[str]:
    # Construye el vocabulario del corpus.
    vocab = set()
    for doc in docs:
        vocab.update(tokenizeText(doc.text))
    return sorted(vocab)


def computeIdf(docs: List[Document], vocabulary: List[str]) -> Dict[str, float]:
    # Calcula IDF suavizado para cada término.
    n_docs = len(docs)
    doc_freq = defaultdict(int)

    for doc in docs:
        unique_tokens = set(tokenizeText(doc.text))
        for token in unique_tokens:
            doc_freq[token] += 1

    idf = {}
    for term in vocabulary:
        idf[term] = math.log((1 + n_docs) / (1 + doc_freq[term])) + 1

    return idf


vocabulary = buildVocabulary(documents)
idf_values = computeIdf(documents, vocabulary)

print("Tamaño del vocabulario:", len(vocabulary))
print("Primeros términos:", vocabulary[:20])

#### **4. Recuperador léxico tipo BM25 simplificado**


BM25 es un recuperador sparse que premia la coincidencia de términos importantes entre la consulta y el documento. Es especialmente útil para nombres propios, siglas, códigos, funciones y frases exactas. En RAG real, BM25 suele complementar embeddings porque los vectores densos pueden perder precisión con entidades o términos raros.

La implementación siguiente no busca ser una réplica completa de BM25 industrial, pero conserva la intuición: frecuencia de término, longitud del documento e importancia del término.

In [ ]:
def bm25Score(query: str, doc: Document, docs: List[Document], k1: float = 1.5, b: float = 0.75) -> float:
    # Calcula un puntaje BM25 simplificado para un documento.
    query_terms = tokenizeText(query)
    doc_terms = tokenizeText(doc.text)
    doc_tf = termFrequency(doc_terms)

    avg_doc_len = sum(len(tokenizeText(d.text)) for d in docs) / max(len(docs), 1)
    doc_len = len(doc_terms)
    score = 0.0

    for term in query_terms:
        if term not in idf_values:
            continue

        tf = doc_tf[term]
        numerator = tf * (k1 + 1)
        denominator = tf + k1 * (1 - b + b * doc_len / max(avg_doc_len, 1))
        score += idf_values[term] * numerator / max(denominator, 1e-9)

    return score


def bm25Retrieve(query: str, docs: List[Document], k: int = 3) -> List[Dict[str, Any]]:
    # Recupera documentos con un puntaje BM25 simplificado.
    scored = []
    for doc in docs:
        scored.append({
            "id": doc.id,
            "title": doc.title,
            "text": doc.text,
            "score": bm25Score(query, doc, docs),
            "retriever": "bm25",
        })

    scored.sort(key=lambda item: item["score"], reverse=True)
    return scored[:k]


bm25_results = bm25Retrieve("¿Qué controlan los guardrails?", documents, k=3)
for item in bm25_results:
    print(item["id"], round(item["score"], 4), item["title"])

#### **5. Recuperador denso simulado con TF-IDF**

Un recuperador denso real transforma la consulta y los documentos en embeddings. Luego busca los vectores más cercanos por similitud coseno, producto punto o distancia L2. Aquí se simula esa idea con vectores TF-IDF para no depender de descargas externas.

Esta simulación permite explicar el flujo de un vector store: convertir documentos a vectores, convertir la consulta a vector, calcular similitud y devolver los top-k. En producción, esta sección se puede reemplazar por `SentenceTransformer`, FAISS o Qdrant.

In [ ]:
def vectorizeText(text: str, vocabulary: List[str], idf: Dict[str, float]) -> List[float]:
    # Convierte texto en un vector TF-IDF simple.
    tokens = tokenizeText(text)
    tf = termFrequency(tokens)
    total = max(len(tokens), 1)

    vector = []
    for term in vocabulary:
        value = (tf[term] / total) * idf.get(term, 0.0)
        vector.append(value)

    return vector


def cosineSimilarity(vec_a: List[float], vec_b: List[float]) -> float:
    # Calcula similitud coseno entre dos vectores.
    dot = sum(a * b for a, b in zip(vec_a, vec_b))
    norm_a = math.sqrt(sum(a * a for a in vec_a))
    norm_b = math.sqrt(sum(b * b for b in vec_b))

    if norm_a == 0 or norm_b == 0:
        return 0.0

    return dot / (norm_a * norm_b)


document_vectors = {
    doc.id: vectorizeText(doc.text, vocabulary, idf_values)
    for doc in documents
}


def denseRetrieve(query: str, docs: List[Document], k: int = 3) -> List[Dict[str, Any]]:
    # Recupera documentos por similitud coseno usando vectores TF-IDF.
    query_vector = vectorizeText(query, vocabulary, idf_values)

    scored = []
    for doc in docs:
        score = cosineSimilarity(query_vector, document_vectors[doc.id])
        scored.append({
            "id": doc.id,
            "title": doc.title,
            "text": doc.text,
            "score": score,
            "retriever": "dense_tfidf",
        })

    scored.sort(key=lambda item: item["score"], reverse=True)
    return scored[:k]


dense_results = denseRetrieve("¿Qué problema resuelve recuperar evidencia externa?", documents, k=3)
for item in dense_results:
    print(item["id"], round(item["score"], 4), item["title"])

#### **6. Búsqueda híbrida: BM25 + recuperación densa**

La búsqueda híbrida fusiona dos señales. BM25 ayuda cuando la consulta contiene palabras exactas. La búsqueda densa ayuda cuando la pregunta usa vocabulario diferente al corpus. En sistemas reales, la fusión puede hacerse con normalización de puntajes, reciprocal rank fusion, modelos de aprendizaje para ranking o rerankers neuronales.

Aquí se normalizan puntajes y se combinan con `alpha`. Si `alpha` es alto, pesa más la señal densa. Si es bajo, pesa más BM25.

In [ ]:
def minMaxNormalize(scores: Dict[str, float]) -> Dict[str, float]:
    # Normaliza puntajes al rango 0 a 1.
    if not scores:
        return {}

    values = list(scores.values())
    min_value = min(values)
    max_value = max(values)

    if max_value == min_value:
        return {key: 0.0 for key in scores}

    return {
        key: (value - min_value) / (max_value - min_value)
        for key, value in scores.items()
    }


def hybridRetrieve(query: str, docs: List[Document], k: int = 3, alpha: float = 0.6) -> List[Dict[str, Any]]:
    # Combina puntajes densos y BM25.
    bm25_all = bm25Retrieve(query, docs, k=len(docs))
    dense_all = denseRetrieve(query, docs, k=len(docs))

    bm25_scores = {item["id"]: item["score"] for item in bm25_all}
    dense_scores = {item["id"]: item["score"] for item in dense_all}

    bm25_norm = minMaxNormalize(bm25_scores)
    dense_norm = minMaxNormalize(dense_scores)

    by_id = {doc.id: doc for doc in docs}
    fused = []

    for doc_id in by_id:
        score = alpha * dense_norm.get(doc_id, 0.0) + (1 - alpha) * bm25_norm.get(doc_id, 0.0)
        doc = by_id[doc_id]
        fused.append({
            "id": doc.id,
            "title": doc.title,
            "text": doc.text,
            "score": score,
            "retriever": "hybrid",
        })

    fused.sort(key=lambda item: item["score"], reverse=True)
    return fused[:k]


hybrid_results = hybridRetrieve("¿Por qué BM25 y embeddings se complementan?", documents, k=4, alpha=0.6)
for item in hybrid_results:
    print(item["id"], round(item["score"], 4), item["title"])

#### **7. Query rewriting y multi-query retrieval**


La expansión o reescritura de consulta ayuda cuando el usuario usa vocabulario distinto al corpus. Un sistema puede generar varias versiones de la pregunta: una literal, una semántica y una técnica. Cada versión recupera documentos, luego se fusionan los resultados.

La ventaja es mayor recall. El riesgo es introducir ruido si las consultas generadas se alejan de la intención original. Por eso esta técnica debe evaluarse.

In [ ]:
def rewriteQuery(question: str) -> List[str]:
    # Genera variantes simples de una consulta en español.
    normalized = normalizeText(question)

    variants = [question]

    replacements = {
        "rag": "recuperación aumentada con generación evidencia documentos externos",
        "embeddings": "vectores densos significado similitud semántica",
        "bm25": "búsqueda por palabras clave coincidencia exacta términos",
        "reranking": "reordenamiento documentos recuperados evidencia relevante",
        "guardrails": "controles seguridad permisos validación salida",
        "agentes": "LLM herramientas memoria planificación verificación",
        "herramientas": "acciones verificables observación agente",
    }

    for key, expansion in replacements.items():
        if key in normalized:
            variants.append(f"{question} {expansion}")

    if len(variants) == 1:
        variants.append(f"{question} evidencia contexto respuesta fundamentada")

    return list(dict.fromkeys(variants))


def multiQueryRetrieve(question: str, docs: List[Document], k_per_query: int = 3, final_k: int = 5) -> List[Dict[str, Any]]:
    # Recupera documentos usando varias consultas y fusiona resultados.
    queries = rewriteQuery(question)
    accumulated: Dict[str, Dict[str, Any]] = {}

    for query in queries:
        results = hybridRetrieve(query, docs, k=k_per_query, alpha=0.6)
        for rank, item in enumerate(results, start=1):
            bonus = 1.0 / rank
            if item["id"] not in accumulated:
                accumulated[item["id"]] = dict(item)
                accumulated[item["id"]]["score"] = 0.0
                accumulated[item["id"]]["queries"] = []

            accumulated[item["id"]]["score"] += bonus
            accumulated[item["id"]]["queries"].append(query)

    fused = list(accumulated.values())
    fused.sort(key=lambda item: item["score"], reverse=True)
    return fused[:final_k]


mq_results = multiQueryRetrieve("¿Qué controlan los guardrails?", documents, k_per_query=3, final_k=5)
for item in mq_results:
    print(item["id"], round(item["score"], 4), item["title"])

#### **8. Reranking y compresión contextual**


El primer retrieval busca candidatos. El reranker reordena esos candidatos con una señal más precisa. En sistemas avanzados, se usa un cross-encoder que evalúa el par pregunta-documento. Aquí se usa un reranker simple por solapamiento de términos importantes.

La compresión contextual reduce el texto antes de enviarlo al LLM. Esto disminuye ruido, tokens y riesgo de que el modelo use evidencia irrelevante.

In [ ]:
def importantTerms(text: str) -> set[str]:
    # Extrae términos simples de longitud suficiente.
    return {token for token in tokenizeText(text) if len(token) >= 5}


def rerankDocuments(question: str, retrieved_docs: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    # Reordena documentos usando solapamiento de términos importantes.
    q_terms = importantTerms(question)
    reranked = []

    for item in retrieved_docs:
        d_terms = importantTerms(item["text"])
        overlap = len(q_terms & d_terms)
        rerank_score = item["score"] + overlap * 0.25

        updated = dict(item)
        updated["rerank_score"] = rerank_score
        updated["term_overlap"] = overlap
        reranked.append(updated)

    reranked.sort(key=lambda item: item["rerank_score"], reverse=True)
    return reranked


def compressContext(question: str, reranked_docs: List[Dict[str, Any]], max_docs: int = 3) -> str:
    # Construye contexto compacto con citas por documento.
    selected = reranked_docs[:max_docs]
    blocks = []

    for item in selected:
        block = f"[{item['id']}] {item['title']}: {item['text']}"
        blocks.append(block)

    return "\n\n".join(blocks)


retrieved = multiQueryRetrieve("¿Para qué sirve el reranking en RAG avanzado?", documents)
reranked = rerankDocuments("¿Para qué sirve el reranking en RAG avanzado?", retrieved)
context = compressContext("¿Para qué sirve el reranking en RAG avanzado?", reranked)

print(context)

#### **9. Generación fundamentada simulada**


En un sistema real, el contexto comprimido se enviaría a un LLM con instrucciones de responder solo con evidencia y citar documentos. Para mantener el cuaderno autocontenido, se implementa un generador extractivo simple. Este generador selecciona oraciones relevantes del contexto y agrega citas.

La finalidad es evaluar el pipeline sin depender de un modelo externo. La celda también muestra cómo debe diseñarse un prompt RAG: contexto, pregunta, restricción de evidencia, formato de citas y regla de abstención.

In [ ]:
def splitSentences(text: str) -> List[str]:
    # Divide texto en oraciones simples.
    parts = re.split(r"(?<=[\.\?\!])\s+", text.strip())
    return [part.strip() for part in parts if part.strip()]


def groundedGenerate(question: str, context: str, max_sentences: int = 3) -> str:
    # Genera una respuesta extractiva basada solo en el contexto.
    q_terms = importantTerms(question)
    candidates = []

    for sentence in splitSentences(context):
        s_terms = importantTerms(sentence)
        score = len(q_terms & s_terms)
        if score > 0:
            candidates.append((score, sentence))

    if not candidates:
        return "No lo sé con la evidencia disponible."

    candidates.sort(key=lambda item: item[0], reverse=True)
    selected = [sentence for _, sentence in candidates[:max_sentences]]

    answer = " ".join(selected)
    return answer


sample_answer = groundedGenerate("¿Qué controlan los guardrails?", context)
print(sample_answer)

#### **10. Guardrails y verificación de fundamentación**


Un guardrail puede validar entradas, herramientas, permisos, formato, citas y salidas. La verificación de fundamentación revisa si la respuesta está apoyada por el contexto. Esta sección implementa controles simples: respuesta no vacía, longitud máxima, ausencia de patrones sensibles, citas válidas y términos obligatorios.

En sistemas reales, esta capa puede incluir validadores JSON Schema, reglas de autorización, detectores de inyección de prompt, verificadores de citas, pruebas de ejecución y evaluadores con LLM.

In [ ]:
def extractCitations(answer: str) -> set[str]:
    # Extrae citas con formato [doc_id].
    return set(re.findall(r"\[(doc_[a-z0-9_]+)\]", answer))


def policyCheck(answer: str, max_chars: int = 900) -> Dict[str, Any]:
    # Aplica guardrails básicos sobre la respuesta final.
    lowered = answer.lower()
    banned_patterns = ["contraseña", "password", "api_key", "token secreto"]

    if not answer.strip():
        return {"passed": False, "reason": "La respuesta está vacía."}

    if len(answer) > max_chars:
        return {"passed": False, "reason": "La respuesta supera la longitud permitida."}

    for pattern in banned_patterns:
        if pattern in lowered:
            return {"passed": False, "reason": f"La respuesta contiene un patrón sensible: {pattern}."}

    return {"passed": True, "reason": "La respuesta cumple los guardrails básicos."}


def verifyGrounding(answer: str, context: str, retrieved_docs: List[Dict[str, Any]], required_terms: List[str]) -> Dict[str, Any]:
    # Verifica citas, términos obligatorios y soporte contextual.
    answer_lower = answer.lower()
    retrieved_ids = {item["id"] for item in retrieved_docs}

    citations = extractCitations(answer)
    invalid_citations = citations - retrieved_ids

    missing_terms = [
        term for term in required_terms
        if term.lower() not in answer_lower
    ]

    context_tokens = set(tokenizeText(context))
    long_terms = [
        token for token in tokenizeText(answer)
        if len(token) >= 9 and token not in context_tokens
    ]

    return {
        "citations": sorted(citations),
        "invalid_citations": sorted(invalid_citations),
        "missing_terms": missing_terms,
        "unsupported_terms": sorted(set(long_terms))[:10],
        "is_grounded": len(invalid_citations) == 0 and len(long_terms) <= 5,
    }


check = policyCheck(sample_answer)
print(check)

#### **11. Pipeline RAG básico**


El RAG básico usa una sola consulta, recupera top-k documentos, concatena el contexto y genera la respuesta. Es simple, barato y fácil de depurar. Su desventaja es que puede fallar cuando la consulta del usuario usa vocabulario distinto, cuando el documento relevante queda fuera del top-k o cuando el contexto contiene ruido.

Este pipeline sirve como baseline. Todo sistema avanzado debe compararse contra una versión simple para justificar su complejidad.

In [ ]:
def simpleRag(question: str, docs: List[Document], k: int = 3) -> Dict[str, Any]:
    # Ejecuta un RAG básico con recuperación híbrida y generación extractiva.
    start = time.perf_counter()

    retrieved = hybridRetrieve(question, docs, k=k, alpha=0.6)
    context = compressContext(question, retrieved, max_docs=k)
    answer = groundedGenerate(question, context)

    elapsed = time.perf_counter() - start

    return {
        "question": question,
        "retrieved_docs": retrieved,
        "context": context,
        "answer": answer,
        "latency_seconds": elapsed,
        "pipeline": "simple_rag",
    }


simple_case = simpleRag("¿Qué problema resuelve RAG?", documents, k=3)
print(simple_case["answer"])

#### **12. Pipeline RAG avanzado**


El RAG avanzado agrega reescritura de consulta, multi-query retrieval, búsqueda híbrida, reranking, compresión contextual, generación fundamentada y verificación. Cada etapa agrega costo y complejidad, pero también puede mejorar recall, orden de evidencia y control de alucinaciones.

El objetivo de esta sección es mostrar que RAG avanzado no es solo agregar más documentos al prompt. Es un pipeline con decisiones explícitas y medibles.

In [ ]:
def advancedRag(question: str, docs: List[Document], required_terms: Optional[List[str]] = None) -> Dict[str, Any]:
    # Ejecuta RAG avanzado con reescritura, fusión, reranking y verificación.
    start = time.perf_counter()

    queries = rewriteQuery(question)
    retrieved = multiQueryRetrieve(question, docs, k_per_query=4, final_k=6)
    reranked = rerankDocuments(question, retrieved)
    context = compressContext(question, reranked, max_docs=3)
    answer = groundedGenerate(question, context, max_sentences=3)

    policy = policyCheck(answer)
    grounding = verifyGrounding(
        answer=answer,
        context=context,
        retrieved_docs=reranked[:3],
        required_terms=required_terms or [],
    )

    elapsed = time.perf_counter() - start

    return {
        "question": question,
        "queries": queries,
        "retrieved_docs": reranked[:3],
        "context": context,
        "answer": answer,
        "policy": policy,
        "grounding": grounding,
        "latency_seconds": elapsed,
        "pipeline": "advanced_rag",
    }


advanced_case = advancedRag(
    "¿Por qué BM25 y embeddings son complementarios?",
    documents,
    required_terms=["BM25", "embeddings", "similitud semántica"],
)

print(advanced_case["answer"])
print(advanced_case["grounding"])

#### **13. Métricas de recuperación**


La evaluación de RAG debe separar recuperación y generación. Si la recuperación falla, el LLM no tiene evidencia. Si la recuperación funciona pero la respuesta falla, el problema está en prompting, compresión, generación o verificación.

Las métricas usadas aquí son `Recall@k`, `Precision@k` y `MRR`. `Recall@k` mide si aparece algún documento correcto. `Precision@k` mide cuánto ruido hay entre los recuperados. `MRR` mide qué tan pronto aparece el primer documento correcto.

In [ ]:
def recallAtK(gold_docs: set[str], retrieved_docs: List[Dict[str, Any]], k: int) -> float:
    # Calcula Recall@k para un caso.
    retrieved_ids = {item["id"] for item in retrieved_docs[:k]}
    return 1.0 if gold_docs & retrieved_ids else 0.0


def precisionAtK(gold_docs: set[str], retrieved_docs: List[Dict[str, Any]], k: int) -> float:
    # Calcula Precision@k para un caso.
    retrieved_ids = [item["id"] for item in retrieved_docs[:k]]
    if not retrieved_ids:
        return 0.0

    hits = sum(1 for doc_id in retrieved_ids if doc_id in gold_docs)
    return hits / len(retrieved_ids)


def reciprocalRank(gold_docs: set[str], retrieved_docs: List[Dict[str, Any]], k: int) -> float:
    # Calcula reciprocal rank para un caso.
    for rank, item in enumerate(retrieved_docs[:k], start=1):
        if item["id"] in gold_docs:
            return 1.0 / rank
    return 0.0

#### **14. Métricas de respuesta**


La respuesta final se evalúa con métricas simples: cobertura de términos requeridos, validez de citas, resultado de guardrails y fundamentación. En un sistema profesional se podría agregar exact match, F1, LLM-as-a-judge, pruebas de factualidad, validación de formato y evaluación humana.

La cobertura de términos no es una métrica perfecta, pero permite detectar respuestas incompletas en un benchmark pequeño.

In [ ]:
def requiredTermCoverage(answer: str, required_terms: List[str]) -> float:
    # Calcula qué proporción de términos requeridos aparece en la respuesta.
    if not required_terms:
        return 1.0

    answer_lower = answer.lower()
    hits = sum(1 for term in required_terms if term.lower() in answer_lower)
    return hits / len(required_terms)


def evaluateCase(item: Dict[str, Any], result: Dict[str, Any], k: int = 3) -> Dict[str, Any]:
    # Evalúa recuperación y respuesta para un caso del benchmark.
    retrieved_docs = result["retrieved_docs"]
    answer = result["answer"]

    policy = result.get("policy", policyCheck(answer))
    grounding = result.get(
        "grounding",
        verifyGrounding(answer, result["context"], retrieved_docs, item["required_terms"]),
    )

    return {
        "pipeline": result["pipeline"],
        "question": item["question"],
        "recall_at_k": recallAtK(item["gold_docs"], retrieved_docs, k),
        "precision_at_k": precisionAtK(item["gold_docs"], retrieved_docs, k),
        "mrr": reciprocalRank(item["gold_docs"], retrieved_docs, k),
        "term_coverage": requiredTermCoverage(answer, item["required_terms"]),
        "policy_passed": policy["passed"],
        "is_grounded": grounding["is_grounded"],
        "latency_seconds": result["latency_seconds"],
        "answer": answer,
        "retrieved_ids": [doc["id"] for doc in retrieved_docs[:k]],
    }

#### **15. Comparación experimental: RAG básico contra RAG avanzado**

Esta sección ejecuta ambos pipelines sobre el mismo benchmark. La comparación permite responder preguntas de ingeniería:

- ¿El RAG avanzado recupera más documentos correctos?
- ¿Aumenta la cobertura de términos relevantes?
- ¿Reduce respuestas no fundamentadas?
- ¿Cuánta latencia adicional introduce?
- ¿La complejidad extra está justificada?

Un sistema serio debe reportar estas métricas antes de afirmar que un pipeline avanzado es mejor.

In [ ]:
def runBenchmark() -> List[Dict[str, Any]]:
    # Ejecuta el benchmark para RAG básico y RAG avanzado.
    rows = []

    for item in benchmark:
        simple_result = simpleRag(item["question"], documents, k=3)
        advanced_result = advancedRag(
            item["question"],
            documents,
            required_terms=item["required_terms"],
        )

        rows.append(evaluateCase(item, simple_result, k=3))
        rows.append(evaluateCase(item, advanced_result, k=3))

    return rows


evaluation_rows = runBenchmark()

for row in evaluation_rows:
    print(
        row["pipeline"],
        "| recall:", row["recall_at_k"],
        "| precision:", round(row["precision_at_k"], 2),
        "| cobertura:", round(row["term_coverage"], 2),
        "| grounded:", row["is_grounded"],
        "| pregunta:", row["question"],
    )

#### **16. Resumen agregado de métricas**


Las métricas por caso son útiles para depurar, pero también se necesita una vista agregada. Promediar métricas ayuda a comparar variantes de pipeline, valores de top-k, pesos de búsqueda híbrida o modelos de embeddings.

En producción, estas métricas deben almacenarse con versión de corpus, versión de embeddings, fecha de ejecución, commit, configuración de `top-k`, modelo generativo y parámetros de decodificación.

In [ ]:
def aggregateMetrics(rows: List[Dict[str, Any]]) -> Dict[str, Dict[str, float]]:
    # Agrega métricas por tipo de pipeline.
    grouped = defaultdict(list)

    for row in rows:
        grouped[row["pipeline"]].append(row)

    summary = {}
    metric_names = [
        "recall_at_k",
        "precision_at_k",
        "mrr",
        "term_coverage",
        "latency_seconds",
    ]

    for pipeline, items in grouped.items():
        summary[pipeline] = {}
        for metric in metric_names:
            summary[pipeline][metric] = sum(item[metric] for item in items) / len(items)

        summary[pipeline]["grounded_rate"] = sum(1 for item in items if item["is_grounded"]) / len(items)
        summary[pipeline]["policy_pass_rate"] = sum(1 for item in items if item["policy_passed"]) / len(items)

    return summary


summary = aggregateMetrics(evaluation_rows)

for pipeline, metrics in summary.items():
    print("\nPipeline:", pipeline)
    for key, value in metrics.items():
        print(f"  {key}: {value:.4f}")

#### **17. Diagnóstico de fallos**


Cuando una respuesta falla, se debe clasificar la causa. Esta separación es central en RAG avanzado:

- Retrieval pobre - no aparece evidencia correcta.
- Contexto ruidoso - aparece demasiada evidencia irrelevante.
- Respuesta no fundamentada - el modelo dice cosas no apoyadas por el contexto.
- Guardrail bloqueante - la política impide entregar la respuesta.
- Cobertura baja - faltan términos esenciales.

Esta clasificación permite mejorar el sistema con precisión. No se corrige igual un problema de chunking que un problema de generación.

In [ ]:
def classifyFailure(row: Dict[str, Any]) -> str:
    # Clasifica el fallo principal de un caso evaluado.
    if row["recall_at_k"] == 0:
        return "retrieval pobre"

    if row["precision_at_k"] < 0.34:
        return "contexto ruidoso"

    if not row["is_grounded"]:
        return "respuesta no fundamentada"

    if not row["policy_passed"]:
        return "guardrail bloqueante"

    if row["term_coverage"] < 0.67:
        return "cobertura baja"

    return "sin fallo crítico"


for row in evaluation_rows:
    failure = classifyFailure(row)
    print(row["pipeline"], "|", failure, "|", row["question"])

#### **18. Análisis de top-k**

`top-k` controla cuántos documentos se recuperan. Un valor bajo puede perder evidencia. Un valor alto puede introducir ruido y aumentar costo. Por eso top-k debe evaluarse, no elegirse arbitrariamente.

Esta sección compara varios valores de top-k en el RAG básico. En un sistema real también se medirían latencia, costo por tokens, calidad de citas y tasa de alucinación.

In [ ]:
def evaluateTopK(k_values: List[int]) -> List[Dict[str, Any]]:
    # Evalúa RAG básico con distintos valores de top-k.
    results = []

    for k in k_values:
        rows = []
        for item in benchmark:
            result = simpleRag(item["question"], documents, k=k)
            rows.append(evaluateCase(item, result, k=k))

        avg_recall = sum(row["recall_at_k"] for row in rows) / len(rows)
        avg_precision = sum(row["precision_at_k"] for row in rows) / len(rows)
        avg_coverage = sum(row["term_coverage"] for row in rows) / len(rows)

        results.append({
            "top_k": k,
            "avg_recall": avg_recall,
            "avg_precision": avg_precision,
            "avg_coverage": avg_coverage,
        })

    return results


topk_results = evaluateTopK([1, 2, 3, 5, 7])

for item in topk_results:
    print(item)

#### **19. Agentic RAG: decisión, acción, observación y parada**

Un RAG clásico siempre sigue el mismo flujo. Un agentic RAG decide si necesita buscar, reformular, calcular, verificar o detenerse. Esta sección implementa una versión mínima: el agente intenta RAG avanzado, verifica si la respuesta tiene cobertura suficiente y solo repite con una consulta expandida si la evidencia no alcanza.

El criterio de parada es esencial. Un agente no debe ejecutar herramientas indefinidamente. Debe detenerse cuando una nueva acción ya no aporta evidencia suficiente o cuando alcanza el máximo de pasos.

In [ ]:
@dataclass
class AgentStep:
    thought: str
    action: str
    observation: str


@dataclass
class AgentTrace:
    task: str
    steps: List[AgentStep] = field(default_factory=list)
    final_answer: str = ""

    def add(self, thought: str, action: str, observation: str) -> None:
        # Agrega un paso a la traza del agente.
        self.steps.append(AgentStep(thought=thought, action=action, observation=observation))


def agenticRag(question: str, docs: List[Document], required_terms: List[str], max_steps: int = 2) -> AgentTrace:
    # Ejecuta un RAG agentic mínimo con verificación y criterio de parada.
    trace = AgentTrace(task=question)
    current_question = question

    for step_index in range(max_steps):
        result = advancedRag(current_question, docs, required_terms=required_terms)
        evaluation = {
            "term_coverage": requiredTermCoverage(result["answer"], required_terms),
            "is_grounded": result["grounding"]["is_grounded"],
            "policy_passed": result["policy"]["passed"],
        }

        trace.add(
            thought="Necesito recuperar evidencia y verificar si la respuesta es suficiente.",
            action=f"advanced_rag paso {step_index + 1}",
            observation=str(evaluation),
        )

        if evaluation["term_coverage"] >= 0.67 and evaluation["is_grounded"] and evaluation["policy_passed"]:
            trace.final_answer = result["answer"]
            trace.add(
                thought="La evidencia y la verificación son suficientes.",
                action="detener",
                observation="No se requieren más acciones.",
            )
            return trace

        current_question = current_question + " evidencia términos clave documentos externos verificación"

    trace.final_answer = result["answer"]
    trace.add(
        thought="Se alcanzó el máximo de pasos permitido.",
        action="detener_por_límite",
        observation="Se entrega la mejor respuesta disponible.",
    )
    return trace


agent_trace = agenticRag(
    "¿Cómo se relacionan agentes y herramientas?",
    documents,
    required_terms=["agente", "herramientas", "observa", "evidencia"],
)

for step in agent_trace.steps:
    print(step.action, "|", step.observation)

print("\nRespuesta final:")
print(agent_trace.final_answer)

#### **20. Discusión técnica final**


Este cuaderno muestra que RAG avanzado no es una sola técnica, sino una arquitectura evaluable. El sistema completo combina recuperación léxica, recuperación densa, fusión, reescritura de consulta, reranking, compresión, generación fundamentada, guardrails, verificación y diagnóstico de fallos.

La conclusión principal es que cada componente tiene una función y también un modo de falla:

- Embeddings - recuperan similitud semántica, pero pueden perder términos exactos.
- BM25 - recupera coincidencias exactas, pero puede fallar con paráfrasis.
- Top-k - controla recall y ruido.
- Reranking - mejora el orden de evidencia.
- Compresión - reduce ruido antes del LLM.
- Guardrails - controlan seguridad, formato y salidas.
- Verificación - comprueba si la respuesta está apoyada por evidencia.
- Evaluación - separa errores de recuperación, contexto y generación.
- Agentic RAG - decide cuándo actuar, verificar y detenerse.

Un sistema profesional no debe afirmar que responde bien solo porque el texto suena convincente. Debe medir recuperación, fidelidad, citas, latencia, costo y seguridad.

#### **21. Ejercicios finales**


Los ejercicios cierran la semana porque obligan a modificar el pipeline, medir efectos y justificar decisiones. La meta es que el estudiante no solo ejecute RAG, sino que pueda diagnosticarlo.

1. Cambia `alpha` en `hybridRetrieve` con valores `0.2`, `0.5` y `0.8`. Explica cuándo mejora BM25 y cuándo mejora la señal densa.
2. Evalúa `top-k` con valores `1`, `3`, `5` y `7`. Reporta recall, precision y cobertura.
3. Agrega un documento distractor sobre un tema parecido y mide si aumenta el ruido.
4. Modifica `rewriteQuery` para generar pseudo consultas. Evalúa si mejora recall o introduce ruido.
5. Implementa un reranker alternativo que premie citas, metadatos o coincidencia de título.
6. Agrega un guardrail que bloquee respuestas sin citas.
7. Agrega un campo `source` al metadata de cada documento y obliga a mostrarlo en la respuesta.
8. Diseña una métrica de `context_noise`.
9. Compara `simpleRag`, `advancedRag` y `agenticRag`.
10. Redacta un análisis de fallos para cada pregunta del benchmark.

## Tus respuestas